# SarcasTone - Notebook 02: Phase 2 (Speech)

Reproduces the acoustic models on the **locked** splits and the one significant contrast.

| Stage | Model | Command |
|---|---|---|
| A | Praat summary (11 feats) + LogReg | `speech_cnn --baseline` |
| B | 1D-CNN over MFCC+deltas+RMS (40 x 200) | `speech_cnn` |
| C | BiGRU over the same sequences | `speech_rnn` |

**Reference (locked test, n=104):** LR 0.6527 | CNN **0.7180** | BiGRU 0.5974.
Only **CNN vs BiGRU** is significant (dF1 +0.12, p=0.0365).

> All inputs needed here (`features_seq/*.npy`, `acoustic_summary.csv`, `acoustic_norm.json`)
> are **committed**, so no audio download is required. Raw audio is only needed to *rebuild*
> features (see `00_setup` step 5).

In [ ]:
import os, sys
PROJECT = '/content/SarcasTone'
if os.path.isdir(PROJECT):
    os.chdir(PROJECT)
sys.path.insert(0, os.path.abspath('src'))
print('cwd =', os.getcwd())

RUN_LR_BASELINE = True
RUN_CNN         = True
RUN_RNN         = True

## 0. Re-derive the locked splits (deterministic, seed 42)

The speech modules call `make_splits()` themselves, which needs the public annotation JSON.

In [ ]:
from sarcastone.data.download_mustard import download_mustard
download_mustard()

# sanity: should print train 482 / val 104 / test 104

In [ ]:
import json
from pathlib import Path
from sarcastone.utils import REPORTS_DIR

before = {}
for t in ('lr', 'cnn', 'rnn'):
    p = Path(REPORTS_DIR) / f'phase2_{t}_test_metrics.json'
    before[t] = json.loads(p.read_text()) if p.exists() else None
    d = before[t]
    print(f'phase2_{t:4s}', (f"F1={d['f1_macro']:.4f}" if d else 'MISSING (expected for rnn)'))

## A. Praat summary + LogReg (weak-modality floor)

11 prosodic/voice features (f0 stats, intensity, HNR, jitter, shimmer). Gate: F1 >= 0.45.

In [ ]:
if RUN_LR_BASELINE:
    !python -m sarcastone.models.speech_cnn --baseline

## B. 1D-CNN over acoustic sequences

Input `(B, 40, T)` z-scored frames; early stopping on val macro-F1 (60 ep max, patience 8).
Dumps penultimate embeddings for Phase 3 (`embeddings/speech_*.npz`).

In [ ]:
if RUN_CNN:
    !python -m sarcastone.models.speech_cnn --extract_embeddings

## C. BiGRU (alternative sequence model)

Same inputs; the weaker of the two sequence models on this data.

In [ ]:
if RUN_RNN:
    !python -m sarcastone.models.speech_rnn

## Verify: fresh vs committed

In [ ]:
import pandas as pd
rows = []
for t in ('lr', 'cnn', 'rnn'):
    p = Path(REPORTS_DIR) / f'phase2_{t}_test_metrics.json'
    a = json.loads(p.read_text()) if p.exists() else None
    b = before.get(t)
    rows.append((t, b['f1_macro'] if b else None, a['f1_macro'] if a else None,
                 round(a['f1_macro'] - b['f1_macro'], 4) if a and b else None))
print(pd.DataFrame(rows, columns=['model', 'committed_F1', 'fresh_F1', 'delta']).to_string(index=False))

## Significance: CNN vs BiGRU (the one significant contrast)

Score both saved checkpoints on the locked test, then exact McNemar + paired bootstrap.
Drops out to a wide CI at n=104 - report the CI, not just the p-value.

In [ ]:
import numpy as np, torch
from torch.utils.data import DataLoader
from sarcastone.data.make_splits import make_splits
from sarcastone.models.speech_data import SeqDataset
from sarcastone.models.speech_cnn import SarcasmCNN
from sarcastone.models.speech_rnn import SarcasmRNN
from sarcastone.training.trainer import evaluate as eval_loop
from sarcastone.evaluation.significance import mcnemar_exact, paired_bootstrap_f1

device = 'cuda' if torch.cuda.is_available() else 'cpu'
splits = make_splits()
test_loader = DataLoader(SeqDataset(splits['test'][['utt_id', 'label']]),
                         batch_size=32, shuffle=False)

probs, ys = {}, None
for name, cls in (('cnn', SarcasmCNN), ('rnn', SarcasmRNN)):
    path = f'checkpoints/speech_{name}/model.pt'
    if not os.path.exists(path):
        print('missing checkpoint:', path); continue
    model = cls().to(device)
    model.load_state_dict(torch.load(path, map_location=device))
    metrics, ys, pr = eval_loop(model, test_loader, device)
    probs[name] = pr
    print(f'{name}: test F1={metrics["f1_macro"]:.4f}  acc={metrics["accuracy"]:.4f}')

In [ ]:
if {'cnn', 'rnn'} <= set(probs):
    pa = (probs['cnn'] >= 0.5).astype(int)
    pb = (probs['rnn'] >= 0.5).astype(int)
    print('McNemar (exact) :', mcnemar_exact(ys, pa, pb))
    print('Bootstrap dF1   :', paired_bootstrap_f1(ys, probs['cnn'], probs['rnn']))

## Reading

- CNN is a viable second modality (F1 ~0.72, above the 0.45 gate); BiGRU underperforms here.
- Only CNN vs BiGRU clears p<0.05 - the LR/CNN gap is not resolvable at n=104.
- These speech embeddings feed Phase 3 fusion, which is **gated** until all Phase 1 & 2
  objectives are closed.